# Modèles de référence et détection
Les calculs sont définis dans `twin/training.py`. Ce notebook affiche les résultats
et les graphiques. Exécution autonome : `python -m twin.training`.
Les cellules d’entraînement et de sauvegarde recréent les modèles locaux.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

root = Path.cwd()
if not (root / "twin").is_dir():
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from twin.training import (
    prepare_data, train_models, evaluate_test,
    calculate_residuals, calibrate_alerts, save_artifacts,
)

In [ ]:
data = prepare_data(root / "outputs/dataset/measurements.csv")
pd.DataFrame({
    "partition": ["train", "validation", "test"],
    "observations": [len(data[key]) for key in ["X_train", "X_val", "X_test"]],
    "variables": [len(data["training_features"])] * 3,
})

## Prévision électrique à une heure avec contexte du bâtiment

La cible est la consommation à l'heure t. Le calendrier décrit t ; les mesures
électriques, météorologiques et de contexte utilisées sont disponibles à t−1
ou avant. Une température extérieure observée à t ne serait pas disponible
lors de la prévision : on conserve `outdoor_temperature_lag_1h_c`.

On inclut l'occupation, la température intérieure et l'énergie thermique HVAC
retardées d'une heure. Ces trois séries sont **simulées** dans ce dataset :
cette expérience évalue leur apport dans ce cadre, sans démontrer un gain sur
des mesures réelles. L'énergie thermique HVAC est distincte de la consommation
électrique cible.

Les retards sont construits avant le filtrage, uniquement si
la ligne précédente correspond à l'heure locale précédente et si les deux
heures ne sont pas signalées comme problématiques pour le changement d'heure.

## Entraînement et comparaison

Les deux modèles utilisent les mêmes 13 variables et les mêmes heures.
Ridge est une régression linéaire régularisée. Ses variables sont standardisées
avec des paramètres appris uniquement sur le train grâce au pipeline.
XGBoost utilise des arbres et ne nécessite pas de standardisation.

On compare les erreurs sur la validation : plus MAE et RMSE sont faibles,
meilleur est le résultat. La persistance (consommation de l'heure précédente)
sert de référence. Le test final est réservé au modèle choisi sur validation.

### 1. Entraînement Ridge et XGBoost

In [ ]:
trained = train_models(data)
ridge_model = trained["ridge_model"]
xgb_model = trained["xgb_model"]

### 3. Comparaison sur validation

In [ ]:
trained["validation_results"]

### 4. Évaluation finale

Le meilleur des deux modèles selon la MAE de validation est évalué sur le test,
sans nouvel ajustement. La persistance reste affichée comme référence.
Après cette évaluation, ne pas utiliser le test pour régler les paramètres.

In [ ]:
evaluate_test(data, trained)

## Détection — étape 1 : figer le modèle de prévision

XGBoost est retenu à partir de la comparaison sur validation. Il reste entraîné
uniquement sur le train : aucun réentraînement sur validation ou test.
La validation servira ensuite à calibrer le détecteur ; le test servira au rejeu.
Les scores du test ayant déjà été consultés, ils ne doivent pas guider les réglages.

On sauvegarde le modèle et l'ordre exact de ses variables pour pouvoir les
recharger lors du rejeu. La sauvegarde est effectuée après la calibration à l’étape 3 ; la relancer remplace les fichiers.
La sauvegarde est locale dans `outputs/models/` (exclu de Git).
Les variables de contexte simulées restent incluses dans cette expérience.

In [ ]:
# Même XGBoost entraîné uniquement sur train. Sauvegarde après calibration.
forecast_model = xgb_model
forecast_features = data["training_features"].copy()

## Détection — étape 2 : analyser les résidus sur validation

Le résidu est la consommation mesurée moins la prédiction : un résidu positif
indique une consommation supérieure à celle prévue, sans constituer à lui seul
une anomalie. On utilise uniquement la validation et le modèle figé.

La moyenne des résidus indique le biais (positive : sous-prédiction en moyenne).
Les quantiles décrivent la distribution des écarts ; aucun seuil d'alerte n'est
choisi à cette étape. Sans annotations, ces écarts ne sont pas certifiés normaux.

In [ ]:
validation_residuals, residual_summary = calculate_residuals(data, forecast_model)
validation_residuals.head()

In [ ]:
residual_summary

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 1, figsize=(12, 7))

# Réintroduire les heures absentes comme NaN pour ne pas relier les lacunes.
residual_timeline = validation_residuals["residual_kwh"].asfreq("h")
axes[0].plot(residual_timeline.index, residual_timeline, linewidth=0.7)
axes[0].axhline(0, color="black", linestyle="--", linewidth=1)
axes[0].set_title("Résidus sur validation — évolution horaire")
axes[0].set_xlabel("Date locale")
axes[0].set_ylabel("Réel − prédit (kWh)")

axes[1].hist(validation_residuals["residual_kwh"], bins=60, edgecolor="white")
axes[1].axvline(0, color="black", linestyle="--", linewidth=1)
axes[1].set_title("Distribution des résidus sur validation")
axes[1].set_xlabel("Réel − prédit (kWh)")
axes[1].set_ylabel("Nombre d'heures")

fig.tight_layout()
plt.show()

## Détection — étape 3 : définir et figer la règle d'alerte

Les seuils sont les quantiles à 1 % (sous-consommation) et à 99 %
(surconsommation) des résidus de validation, sans arrondi.
On compare un dépassement immédiat et deux dépassements horaires consécutifs.
Une heure absente interrompt la séquence. Un épisode regroupe les heures
consécutives où la règle est active ; le début correspond à l'heure où
l'alerte devient disponible, sans anticipation.

On retient deux heures consécutives du même côté du seuil pour écarter les dépassements isolés.
C'est un choix opérationnel de départ : il retarde l'alerte d'une heure et
ne détecte pas les événements d'une seule heure. Sans annotations, cette
comparaison mesure la fréquence des alertes, pas leur précision ni leur rappel.
Le test n'intervient pas dans le choix du seuil ou de la règle.
Une alternance sous-consommation / surconsommation ne déclenche pas une
alerte persistante. Une sous-consommation peut signaler un arrêt inhabituel,
une erreur de mesure ou une erreur de prévision ; elle ne prouve pas un défaut.

In [ ]:
validation_alerts, alert_comparison, alert_rule = calibrate_alerts(validation_residuals)
print("Seuil inférieur :", alert_rule["rules"]["underconsumption"]["threshold_kwh"])
print("Seuil supérieur :", alert_rule["rules"]["overconsumption"]["threshold_kwh"])
alert_comparison

In [ ]:
# Examiner les heures signalées, avec leur type d'alerte.
validation_alerts.loc[
    validation_alerts["under_2h"] | validation_alerts["over_2h"],
    ["actual_kwh", "predicted_kwh", "residual_kwh", "alert_type"],
]

In [ ]:
model_path, rule_path = save_artifacts(
    data, forecast_model, alert_rule, root / "outputs/models",
)
print(f"Modèle : {model_path}")
print(f"Règle : {rule_path}")